# Глубинное обучение для текстовых данных

## Домашнее задание 1: Токенизация и классификация рекуррентными нейронными сетями

### Оценивание и штрафы

Максимально допустимая оценка за работу — 11 баллов. Сдавать задание после указанного срока сдачи нельзя.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов. Весь код должен быть написан самостоятельно. Чужим кодом для пользоваться запрещается даже с указанием ссылки на источник. В разумных рамках, конечно. Взять пару очевидных строчек кода для реализации какого-то небольшого функционала можно.

Неэффективная реализация кода может негативно отразиться на оценке. Также оценка может быть снижена за плохо читаемый код и плохо оформленные графики. Все ответы должны сопровождаться кодом или комментариями о том, как они были получены.

### О задании

Это задание посвящено решению задачи классификации тональности текста с помощью рекуррентных нейронных сетей (и сверточных). В отличие от предыдущей домашки, в этом задании не будет шаблонов кода, вам придется написать все самостоятельно, поэтому постарайтесь хорошо организовать код. Мы крайне рекомендуем реализовывать все модели и вспомогательные функции в отдельных файлах, а затем их импортировать. Иначе вы рискуете превратить ноутбук в кашу.

Обучение не должно занимать много времени, однако для ускорения мы советуем использовать различные приемы, такие как mixed-precision ([ссылка](https://huggingface.co/docs/accelerate/package_reference/accelerator) и [ссылка](https://pytorch.org/docs/stable/amp.html)) и torch.compile (для torch >= 2.0, [ссылка](https://pytorch.org/tutorials/intermediate/torch_compile_tutorial.html)). Так же мы настоятельно рекомендуем логировать все графики обучения с помощью платформы [wandb](https://docs.wandb.ai/quickstart).

Все модели нужно будет реализовывать на pytorch (считать градиенты руками не надо). При обучении разных моделей используйте одинаковые параметры оптимизатора, базмер батча, а так же учите либо одинаковое число эпох, либо до сходимости. В общем, постарайтесь, чтобы сравнение всегда было максимально честным.


__Мягкий дедлайн: 26.10.23 23:59__

__Жесткий дедлайн: 09.11.24 23:59__

### Данные и токенизация

Мы будем обучать модель на задачу классификации тональности текста. Данные взяты с платформы IMDb и содержат положительные и отрицательные отзывы о фильмах, примерно по 5к текстов каждого класса. Вы можете найти их в папке `datasets/IMDb`.

In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


__Задание 1 (2 балла)__
Как всегда, для работы с текстом нам нужно как-то его токенизировать. В этом задании вам предстоит заняться этим. Выберите один из трех методов токенизации, разобранных на лекции: __BPE__, __WordPiece__, __Unigram__. Реализуйте его самостоятельно без использования специализированных библиотек (например, huggingface). Ваш токенизатор должен иметь интерфейс токенизатора из huggingface. То есть он должен иметь метод `encode`, возвращающий словарь с полями `input_ids` и `attention_mask`, а так же метод `decode`, принимающий последовательности токенов и возвращающий соответствующие им тексты. Ограничьте размер словаря 30000 токенами. Ваш токенизатор не обязан добавлять токены начала и конца последовательности, потому что они не обязательны в задаче классификации. Однако при желании вы можете их добавить. Все тонкости реализации, такие как прочие аргументы функций, типы переменных и так далее остаются на ваше усмотрение.

In [ ]:
from bpe_tokenizer import BPETokenizer

In [ ]:
from dataset import IMDBDataset
import os
from torch.utils.data import Dataset, DataLoader, Subset
import torch


__Задание 1 (0 баллов)__
Прочитайте датасет и сложите все в `DataLoader`. Вы можете предобрабатывать тексты дополнительно как вам хочется.

In [ ]:
def create_dataloaders(positive_path: str, negative_path: str, tokenizer, batch_size: int = 32, max_length: int = 512,
                      test_split: float = 0.2, shuffle: bool = True):
    full_dataset = IMDBDataset(positive_path, negative_path, tokenizer, max_length)
    indices = torch.randperm(len(full_dataset)).tolist()
    train_dataset = Subset(full_dataset, indices)
    train_loader = DataLoader(
        full_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )
    return train_loader

def train_tokenizer( positive_path, negative_path, vocab_size: int = 30000, num_merges: int = 10000):
    texts = []
    tokenizer = BPETokenizer(vocab_size=vocab_size)
    with open(positive_path, 'r', encoding='utf-8') as f:
        texts.append(f.read())
    with open(negative_path, 'r', encoding='utf-8') as f:
        texts.append(f.read())
    tokenizer.train(texts, num_merges)
    return tokenizer

## Сверточные нейронные сети

__Задание 3 (2 балла)__ В качестве бейзлайна для решения задачи реализуйте и обучите сверточную нейронную сеть с одномерными свертками (`torch.nn.Conv1d`). Число сверточных слоев можете выбрать по своему усмотрению, однако не делайте сеть слишком большой. Это задание требуется для установки бейзлайна, поэтому лучше не тратить много времени на обучение и подгонку параметров. У вас должна получиться точность классификации на тестовой выборке около 80% (больше тоже хорошо).

In [ ]:
# your code here
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

from CNN import TextCNN

positive_path = "positive_t.txt"
negative_path = "negative_t.txt"
positive_path_test = "positive.txt"
negative_path_test = "negative.txt"

tokenizer = train_tokenizer(positive_path, negative_path, vocab_size=10000, num_merges=10000)

train_loader = create_dataloaders(
    positive_path=positive_path,
    negative_path=negative_path,
    tokenizer=tokenizer,
    batch_size=128,
    max_length=256
)

test_loader = create_dataloaders(
    positive_path=positive_path_test,
    negative_path=negative_path_test,
    tokenizer=tokenizer,
    batch_size=128,
    max_length=256
)


In [ ]:
def train_model(model, train_loader, test_loader, epochs=5, lr=0.001):
      device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
      model = model.to(device)

      criterion = nn.CrossEntropyLoss()
      optimizer = torch.optim.Adam(model.parameters(), lr=lr)

      for epoch in range(epochs):
          model.train()
          total_loss = 0
          correct = 0
          total = 0

          for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}'):
              input_ids = batch['input_ids'].to(device)
              labels = batch['labels'].to(device)
              optimizer.zero_grad()
              outputs = model(input_ids)
              loss = criterion(outputs, labels)
              loss.backward()
              optimizer.step()

              total_loss += loss.item()
              _, predicted = torch.max(outputs.data, 1)
              total += labels.size(0)
              correct += (predicted == labels).sum().item()

          train_loss = total_loss / len(train_loader)
          train_acc = correct / total

          test_acc = evaluate_model(model, test_loader, device)

          print(f'Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}')

def evaluate_model(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return correct / total

In [ ]:
from CNN import TextCNN
vocab_size = len(tokenizer.token_to_idx)
model = TextCNN(vocab_size, embedding_dim=100)
print(f"Model has {sum(p.numel() for p in model.parameters()):,} parameters")

print("Training model...")
train_model(model, train_loader, test_loader, epochs=12, lr=0.001)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
final_acc = evaluate_model(model, test_loader, device)
print(f"\nFinal Test Accuracy: {final_acc:.4f}")

Model has 5,146,702 parameters
Training model...


Epoch 1: 100%|██████████| 77/77 [00:21<00:00,  3.53it/s]


Epoch 1: Train Loss: 0.7230, Train Acc: 0.5667, Test Acc: 0.7698


Epoch 2: 100%|██████████| 77/77 [00:20<00:00,  3.82it/s]


Epoch 2: Train Loss: 0.5382, Train Acc: 0.7261, Test Acc: 0.7984


...

Epoch 10: Train Loss: 0.1676, Train Acc: 0.9331, Test Acc: 0.8647


Epoch 11: 100%|██████████| 77/77 [00:19<00:00,  3.92it/s]


Epoch 11: Train Loss: 0.1558, Train Acc: 0.9380, Test Acc: 0.8617


Epoch 12: 100%|██████████| 77/77 [00:20<00:00,  3.81it/s]


Epoch 12: Train Loss: 0.1304, Train Acc: 0.9514, Test Acc: 0.8659

Final Test Accuracy: 0.8659


## Рекуррентные нейронные сети


В этой секции вам предстоит реализовать два вида рекуррентных нейронных сетей: RNN, LSTM. Вот они с слева направо. Начнем с RNN.

__Задание 4 (1.5 балла)__ Реализуйте классическую рекуррентную нейронную сеть с одним слоем. Как вы знаете из лекции, такая сеть плохо выучивает долгосточные зависимости в данных, поэтому работают плохо с длинными текстами. Обучите реализованную RNN и проверьте, так ли это, замерив точность на тестовой выборке.

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

from RNN import RNN

In [4]:
vocab_size = len(tokenizer.token_to_idx)
model = RNN(vocab_size, embedding_dim=128, hidden_dim=256)

train_model(model, train_loader, test_loader, epochs=10, lr=0.001)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
final_acc = evaluate_model(model, test_loader, device)
print(f"\nFinal Test Accuracy: {final_acc:.4f}")

NameError: name 'tokenizer' is not defined

__Задание 5 (2.5 балла)__
В предыдущем задании скорее всего качество RNN у вас должно было получиться хуже, чем у CNN. Попробуем исправить это, добавив в модель вектор памяти. Реализуйте модель LSTM и сравните ее качество с RNN. Напомним, что скрытые состояния LSTM считаются по следующим формулам:
$$
\begin{gathered}
f_t=\sigma\left(W_f \cdot\left[h_{t-1}, x_t\right]+b_f\right) \\
i_t=\sigma\left(W_i \cdot\left[h_{t-1}, x_t\right]+b_i\right) \\
o_t=\sigma\left(W_o \cdot\left[h_{t-1}, x_t\right]+b_o\right) \\
\tilde{C}_t=\tanh \left(W_c \cdot\left[h_{t-1}, x_t\right]+b_c\right) \\
C_t=f_t \odot C_{t-1}+i_t \odot \tilde{C}_t \\
h_t=o_t \odot \tanh \left(C_t\right)
\end{gathered}
$$

In [ ]:
from LSTM import LSTM
model = LSTM(vocab_size, embedding_dim=128, hidden_dim=256)

train_model(model, train_loader, test_loader, epochs=13, lr=0.002)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
final_acc = evaluate_model(model, test_loader, device)
print(f"\nFinal Test Accuracy: {final_acc:.4f}")

Epoch 1: 100%|██████████| 77/77 [00:51<00:00,  1.49it/s]


Epoch 1: Train Loss: 0.6743, Train Acc: 0.5865, Test Acc: 0.6292


Epoch 2: 100%|██████████| 77/77 [00:51<00:00,  1.49it/s]


Epoch 2: Train Loss: 0.5506, Train Acc: 0.7254, Test Acc: 0.7553


Epoch 3: 100%|██████████| 77/77 [00:52<00:00,  1.48it/s]


...

Epoch 11: Train Loss: 0.0236, Train Acc: 0.9935, Test Acc: 0.8065


Epoch 12: 100%|██████████| 77/77 [00:52<00:00,  1.47it/s]


Epoch 12: Train Loss: 0.0126, Train Acc: 0.9967, Test Acc: 0.8063


Epoch 13: 100%|██████████| 77/77 [00:51<00:00,  1.49it/s]


Epoch 13: Train Loss: 0.0040, Train Acc: 0.9992, Test Acc: 0.8209

Final Test Accuracy: 0.8209


__Бонус (1 балл)__ Проверьте, на что влияет каждый гейт в LSTM. Попробуйте отключать каждый из них по очереди (выход гейта всегда 1) и посмотрите, при отключении каких из них точность падает сильнее. Совпадает ли результат с вашими ожиданиями?

In [ ]:
model.gate_f = False
final_acc = evaluate_model(model, test_loader, device)
print(f"\nFinal Test Accuracy: {final_acc:.4f}")
model.gate_f = True
model.gate_i = False
final_acc = evaluate_model(model, test_loader, device)
print(f"\nFinal Test Accuracy: {final_acc:.4f}")
model.gate_i = True
model.gate_o = False
final_acc = evaluate_model(model, test_loader, device)
print(f"\nFinal Test Accuracy: {final_acc:.4f}")
model.gate_o = True


Final Test Accuracy: 0.6356

Final Test Accuracy: 0.7888

Final Test Accuracy: 0.7088


__Задание 6 (1 балл): Многослойная рекуррентная сеть.__
Часто увеличение число слоев в рекуррентных нейронных сетях помогает улучшить способность модели извлекать информацию из текста. Дополнительный слой – это еще одна LSTM поверх выходов предыдущей. Добавьте в вашу модель возможность создания сети с произвольным числом слоев. Обучите двухслойную LSTM и сравните качество с однослойной версией. Подтвердилась ли теория?



In [ ]:
from LSTM import MultiLayerLSTM
model_mult = MultiLayerLSTM(vocab_size,num_layers=2, embedding_dim=128, hidden_dim=256)

train_model(model_mult, train_loader, test_loader, epochs=8, lr=0.001)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
final_acc = evaluate_model(model_mult, test_loader, device)
print(f"\nFinal Test Accuracy: {final_acc:.4f}")

Epoch 1: 100%|██████████| 77/77 [01:23<00:00,  1.09s/it]


Epoch 1: Train Loss: 0.6619, Train Acc: 0.5976, Test Acc: 0.6559


Epoch 2: 100%|██████████| 77/77 [01:23<00:00,  1.08s/it]


Epoch 2: Train Loss: 0.5790, Train Acc: 0.7009, Test Acc: 0.7016


Epoch 3: 100%|██████████| 77/77 [01:23<00:00,  1.08s/it]


...

Epoch 6: Train Loss: 0.2501, Train Acc: 0.9037, Test Acc: 0.7963


Epoch 7: 100%|██████████| 77/77 [01:22<00:00,  1.07s/it]


Epoch 7: Train Loss: 0.1837, Train Acc: 0.9338, Test Acc: 0.8045


Epoch 8: 100%|██████████| 77/77 [01:22<00:00,  1.08s/it]


Epoch 8: Train Loss: 0.1170, Train Acc: 0.9580, Test Acc: 0.7977

Final Test Accuracy: 0.7977


__Задание 6 (1 балл): Двунаправленная рекуррентная сеть.__ Для некоторых задач классификации может помочь смотреть на последовательность слева направо и справа налево одновременно. Рекуррентные сети с такой способностью называются двунаправленными (bidirectional). Реализуйте данный функционал в вашей LSTM модели. Заметьте, что такая модификация увеличивает число параметров модели так же, как и добавление еще одного слоя. Какая из этих двух модификаций оказывается лучше?

__ВАЖНО:__ Подумайте о том, как лучше агрегировать выходы разнонаправленных моделей. Попробуйте разные способы.


In [ ]:
from LSTM import BiLSTM
model_bi = BiLSTM(vocab_size, embedding_dim=128, hidden_dim=256)

train_model(model_bi, train_loader, test_loader, epochs=5, lr=0.001)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
final_acc = evaluate_model(model_bi, test_loader, device)
print(f"\nFinal Test Accuracy: {final_acc:.4f}")

Epoch 1: 100%|██████████| 77/77 [01:29<00:00,  1.16s/it]


Epoch 1: Train Loss: 0.6450, Train Acc: 0.6163, Test Acc: 0.6930


Epoch 2: 100%|██████████| 77/77 [01:28<00:00,  1.15s/it]


Epoch 2: Train Loss: 0.5129, Train Acc: 0.7450, Test Acc: 0.7614


Epoch 3: 100%|██████████| 77/77 [01:28<00:00,  1.15s/it]


...

Epoch 4: 100%|██████████| 77/77 [01:28<00:00,  1.15s/it]


Epoch 4: Train Loss: 0.3483, Train Acc: 0.8475, Test Acc: 0.7887


Epoch 5: 100%|██████████| 77/77 [01:28<00:00,  1.15s/it]


Epoch 5: Train Loss: 0.2273, Train Acc: 0.9086, Test Acc: 0.8030


NameError: name 'v' is not defined

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
final_acc = evaluate_model(model_bi, test_loader, device)
print(f"\nFinal Test Accuracy: {final_acc:.4f}")


Final Test Accuracy: 0.8030


Опишите тут все, что попробовали. Что получилось, а что нет? Не нужно очень подробных разъяснений, достаточно 2-3 предложения по каждому пункту.

__ВАШ ОТВЕТ__: Пробовал разные batch_size, разные dropout для обучения RNN. Но больший accuracy выбить не смог.